### 模型二：

In [ ]:
import pandas as pd
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, LpBinary, LpStatus
import collections
from itertools import product

# 1. Load data
file_path = "realdata_v5_1_new.xlsx"  # Please replace with your file path
activities_df = pd.read_excel(file_path, sheet_name="ActivitiesInfo")
classroom_df = pd.read_excel(file_path, sheet_name="ClassroomsInfo")
student_courses_df = pd.read_excel(file_path, sheet_name="StudentsInfo")

In [21]:
df = pd.read_csv("processed_schedule.csv")
# df = df[(df["Week"] == 11) & (df["Time_slot"] == 2)]

df

,Week,Time_slot,subject_candidates
0,1,1,"{16, 52}"
1,1,2,"{25, 18, 5, 15}"
2,1,3,"{11, 13}"
3,1,4,"{28, 30}"
4,1,5,{45}
...,...,...,...
90,11,9,{7}
91,12,1,{2}
92,12,3,{33}
93,12,5,{6}


In [22]:
courses = student_courses_df["Course_ID"]
courses = set([int(num) for row in courses for num in row.split(', ')])

classrooms = classroom_df["Classroom_ID"]
classrooms = set(int(row) for row in classrooms)

activity_types = activities_df["Activity_Type"]
activity_types = set(activity_types)

In [ ]:
# 模型二的很多参数是由模型一的参数缩小维度而来，比如模型一有一个参数：Ysai，模型二有一个参数Ysi，去掉了a这个维度
# 参数
Pc = [row["Classroom_ID"] for _, row in classroom_df.iterrows() if row["Has_Computers"] == 1]
Ysi = {(row["Course_ID"], row["Activity_Type"]): 1 for _, row in activities_df.iterrows()}
Gs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Separation"] == 1}
Cs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Computers"] == 1}
Ts = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Tables"] == 1}
Tc = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Has_Tables"] == 1}
Act = {(row["Classroom_ID"], int(w), int(t)): 1 for _, row in classroom_df.iterrows() for w in row["Available_Weeks"].split(', ') for t in row["Time_Slot"].split(', ')} #
Ic = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Is_Isolated"] == 1}
Ns = {row["Course_ID"]: row["Num_Students"] for _, row in activities_df.iterrows()}

Msg = collections.defaultdict(int)
for _, row in student_courses_df.iterrows():
    class_id = row["Class_ID"]
    course_ids = list(map(int, str(row["Course_ID"]).split(', ')))  
    for course_id in course_ids:
        Msg[(course_id, class_id)] += 1  

            
class_courses = collections.defaultdict(set)
for (course, class_id), _ in Msg.items():
    class_courses[course].add(class_id)

Capacity = {row["Classroom_ID"]: row["Capacity"] for _, row in classroom_df.iterrows()}
Occupancy_rate = {1: 1, 2: 0.9, 3: 0}
CAPci = {(c, i): int(Capacity[c] * Occupancy_rate[i]) for c, i in product(classrooms, activity_types)}

In [24]:
import csv
with open("results.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Week", "Time_slot", "subject_candidates", "result","value"])

results = []

In [ ]:


for count in range(len(df)):
    week = df.iloc[count]["Week"]
    time_slot = df.iloc[count]["Time_slot"]
    subjects = df["subject_candidates"].tolist()
    subjects = [int(i.strip('{}')) for i in subjects[count].split(', ')]
    # 2. Define the problem
    model = LpProblem(name="classroom_assignment", sense=LpMinimize)

    z = {(c, s): LpVariable(f"z_{c}_{s}", cat=LpBinary) for c in classrooms for s in subjects}
    w = {(c, s, g): LpVariable(f"w_{c}_{s}_{g}", cat=LpBinary) for c in classrooms for s in subjects for g in class_courses[s]}

    # Objective function
    model += lpSum(z[c, s] for c in classrooms for s in subjects), "Minimize_Classroom_Usage"

    # A.13 - Ensure classroom capacity is sufficient
    for s in subjects:
        model += lpSum(z[c, s] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Ns[s], f"Capacity_Constraint_{s}"

    # A.14 - Ensure classroom capacity is sufficient for each class
    for s in subjects:
        if Ts.get(s, 0) == 1:
            for g in class_courses[s]:
                model += lpSum(w[c, s, g] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Msg.get((s, g), 0), f"Classroom_Capacity_{s}_{g}"

    # A.15 - Prevent classrooms from being reused
    for c in classrooms:
        model += lpSum(z[c, s] for s in subjects) <= 1, f"Single_Assignment_{c}"

    # A.16 - Only allow available classrooms to be assigned
    for c in classrooms:
        for s in subjects:
            model += z[c, s] <= Act.get((c, week, time_slot), 0), f"Classroom_Availability_{c}_{s}"

    # A.17 - Consistency in time slot assignment
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                for g in class_courses[s]:
                    model += z[c, s] >= w[c, s, g], f"Consistency_1_{c}_{s}_{g}"

    # A.18 - Consistency in time slot assignment
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                model += z[c, s] <= lpSum(w[c, s, g] for g in class_courses[s]), f"Consistency_2_{c}_{s}"

    # A.19 - Ensure different classes do not share classrooms
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                for g1 in class_courses[s]:
                    for g2 in class_courses[s]:
                        if g1 != g2:
                            model += w[c, s, g1] + w[c, s, g2] <= 1, f"No_Shared_Classroom_{c}_{s}_{g1}_{g2}"

    # A.20 - Computer room requirements
    for c in classrooms:
        for s in subjects:
            if Cs.get(s, 0) == 1 and c not in Pc:
                model += z[c, s] == 0, f"Computer_Requirement_{c}_{s}"

    # A.21 - Desk requirements
    for c in classrooms:
        for s in subjects:
            if Ts.get(s, 0) == 1 and Tc.get(c, 0) == 0:  # Course requires desks but classroom does not have them
                model += z[c, s] == 0, f"Table_Requirement_{c}_{s}"

    # A.22 - Isolated classrooms
    for c in classrooms:
        for s in subjects:
            if Ic.get(c, 0) == 1:
                model += lpSum(z[c_prime, s] for c_prime in classrooms if c_prime != c) <= (1 - z[c, s]) * len(classrooms), f"Isolation_Requirement_{c}_{s}"

    # 3. Solve the problem
    model.solve()

    # Output optimization results
    print("Week:", week)
    print("Time Slot:", time_slot)
    print("Optimization Status:", LpStatus[model.status])
    print("Objective Function Value:", model.objective.value())
    if model.status == 1:  # Output assignment details only if a feasible solution is found
        print("\nClassroom Assignment Details:")
        assigned_classrooms = []
        for (c, s), var in z.items():
            if var.value() == 1:
                assigned_classrooms.append((c, s))
                print(f" - Classroom {c} assigned to course {s}")
    else:
        print("\n!! No feasible solution found, please check the constraints !!")
        # Output conflicting constraints (if any)
        for name, constraint in model.constraints.items():
            if not constraint.valid():
                print(f"Conflicting Constraint: {name}")

    # Collect and write results within the loop
    if model.status == 1:
        # Collect results for the current iteration
        value = model.objective.value()
        assignment_pairs = []
        for (c, s), var in z.items():
            if var.value() == 1:
                # Sort by course ascending, classroom ascending
                assignment_pairs.append((s, c))
        
        # Generate the result string
        sorted_assignments = sorted(assignment_pairs, key=lambda x: (int(x[0]), int(x[1])))
        result_entries = [f"{{{s}:{c}}}" for s, c in sorted_assignments]
        result_str = ",".join(result_entries)

        # Generate subject_candidates (original logic retained)
        subject_set = {s for s, _ in sorted_assignments}
        sorted_subjects = sorted(subject_set, key=int)
        subject_candidates = "{%s}" % ",".join(map(str, sorted_subjects))
    else:
        subject_candidates = "{}"
        result_str = "{}"

    # Write to CSV (retain original writing logic)
    with open("results.csv", "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            df.iloc[count]["Week"],
            df.iloc[count]["Time_slot"],
            df.iloc[count]["subject_candidates"],
            result_str,
            value
        ])




Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/arm64/cbc /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/c31c3de557494124826a734b8333a652-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/c31c3de557494124826a734b8333a652-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 716 COLUMNS
At line 8617 RHS
At line 9329 BOUNDS
At line 9615 ENDATA
Problem MODEL has 711 rows, 285 columns and 7216 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0.139037 - 0.00 seconds
Cgl0002I 84 variables fixed
Cgl0003I 43 fixed, 0 tightened bounds, 42 strengthened rows, 7 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 0 strengthened rows, 1 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 11 strengthened rows, 0 substitutions
